 

Đánh giá dữ liệu ở lớp Bronze, làm sạch và sử  lý các cột để  dễ  lưu trữ và sử  dụng hơn. 

In [28]:
# khởi taọ và cấu hình cho spark  

import os 
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

from pyspark.sql import SparkSession 
from pyspark.sql import functions as f 
from pyspark.sql.window import Window 

# local[*]: dùng tất cả CPU cores trên máy host, không cần cluster
# Phù hợp cho explore/notebook với data nhỏ như VN30
spark = SparkSession.builder \
    .appName('bronze_ingestion') \
    .master('local[*]') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()



In [29]:
# Đọc tất cả file CSV từ HDFS bronze layer

HDFS_BRONZE = 'hdfs://namenode:9000/user/vn30/bronze/*.csv'

df = spark.read \
    .option('header', 'true') \
    .csv(HDFS_BRONZE)

print(f'Tổng số dòng: {df.count()}')
print(f'Số partitions: {df.rdd.getNumPartitions()}')
df.printSchema()
df.show(5)


Tổng số dòng: 47082
Số partitions: 15
root
 |-- time: string (nullable = true)
 |-- open: string (nullable = true)
 |-- high: string (nullable = true)
 |-- low: string (nullable = true)
 |-- close: string (nullable = true)
 |-- volume: string (nullable = true)

+----------+-----+-----+-----+-----+-------+
|      time| open| high|  low|close| volume|
+----------+-----+-----+-----+-----+-------+
|2019-09-18|20.92|20.92|20.34|20.38|2592910|
|2019-09-19|20.45|20.88| 20.3|20.88|1432730|
|2019-09-20|20.85|21.24|20.81|21.03|1396120|
|2019-09-23|20.95|21.06|20.41|20.41|2313090|
|2019-09-24|20.38|20.59|20.16| 20.3|2537600|
+----------+-----+-----+-----+-----+-------+
only showing top 5 rows



In [30]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import DoubleType, LongType

df_clean = data_raw \
    .withColumn("time", to_date(col("time"), "yyyy-MM-dd")) \
    .withColumn("open", col("open").cast(DoubleType())) \
    .withColumn("high", col("high").cast(DoubleType())) \
    .withColumn("low", col("low").cast(DoubleType())) \
    .withColumn("close", col("close").cast(DoubleType())) \
    .withColumn("volume", col("volume").cast(LongType()))

df_clean.printSchema()


root
 |-- time: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)



In [31]:
# Đếm null từng cột
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

data_raw.select([
    spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c)
    for c in data_raw.columns
]).show()


+----+----+----+---+-----+------+
|time|open|high|low|close|volume|
+----+----+----+---+-----+------+
|   0|   0|   0|  0|    0|     0|
+----+----+----+---+-----+------+



In [32]:
# Tổng số dòng vs số dòng distinct
total = data_raw.count()
distinct = data_raw.distinct().count()
print(f"Tổng dòng: {total}")
print(f"Distinct: {distinct}")
print(f"Trùng: {total - distinct}")

# Xem các dòng bị trùng
from pyspark.sql.functions import count

data_raw.groupBy(data_raw.columns) \
    .agg(count("*").alias("so_lan")) \
    .filter("so_lan > 1") \
    .orderBy("so_lan", ascending=False) \
    .show()


Tổng dòng: 47082
Distinct: 47082
Trùng: 0
+----+----+----+---+-----+------+------+
|time|open|high|low|close|volume|so_lan|
+----+----+----+---+-----+------+------+
+----+----+----+---+-----+------+------+



In [34]:
# Giá âm hoặc bằng 0
data_raw.filter("close <= 0 OR open <= 0 OR high <= 0 OR low <= 0").show()

# Volume âm
data_raw.filter("volume < 0").show()

# high < low (vô lý)
data_raw.filter("high < low").show()

# close nằm ngoài [low, high]
data_raw.filter(col("close") > col("high")).show()
data_raw.filter(col("close") < col("low")).show()
data_raw.filter(col("high") < col("low")).show()


+----+----+----+---+-----+------+
|time|open|high|low|close|volume|
+----+----+----+---+-----+------+
+----+----+----+---+-----+------+

+----+----+----+---+-----+------+
|time|open|high|low|close|volume|
+----+----+----+---+-----+------+
+----+----+----+---+-----+------+

+----------+------+------+-----+------+--------+
|      time|  open|  high|  low| close|  volume|
+----------+------+------+-----+------+--------+
|2024-05-16| 99.24|100.48|98.21| 99.09| 3128143|
|2024-05-22|100.92|102.24| 99.6| 101.0| 4522916|
|2024-05-23|100.63|100.85| 98.8|100.63| 3630410|
|2024-05-24| 101.0| 101.0|95.58| 96.53|13398023|
|2024-05-28| 98.07|100.26|97.41|100.26| 4632228|
|2024-05-29|100.34|101.14|98.65| 99.31| 5844515|
|2024-05-31|  98.8|100.12|97.92| 98.51| 2639516|
|2024-06-03|  98.8| 100.7|98.58|100.19| 7782256|
|2024-08-05|102.62|104.15|99.49|100.51|10450171|
|2025-04-03| 97.88|100.35|96.85| 96.85|11536434|
|2025-04-11| 99.84|101.12|97.54|101.12|19273397|
|2025-04-14| 102.4| 102.4|97.28|101.12| 